# NFL Trading Feature Engineering Demo

This notebook demonstrates the comprehensive feature engineering system for NFL trading, including:
- Loading and processing NFL and Kalshi price data
- Feature extraction using GameStateExtractor, TechnicalIndicators, and MomentumDetector
- Correlation analysis between features and price movements
- Feature validation and visualization
- Feature importance rankings

We'll use the Dallas Cowboys at Philadelphia Eagles game data as an example.

In [ ]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Import our feature engineering modules
from src.nfl_trading.features import (
    FeaturePipeline, 
    FeatureConfig, 
    FeatureValidator, 
    FeatureVisualizer,
    GameStateExtractor,
    TechnicalIndicators,
    MomentumDetector
)

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Data Loading and Preprocessing

First, let's load the sample game data. In a real scenario, you would load this from your data sources.

In [ ]:
def create_sample_nfl_data():
    """Create realistic sample NFL play-by-play data."""
    base_time = datetime(2025, 9, 4, 20, 20)  # Game start time
    
    # Simulate a realistic game progression
    plays = []
    current_time = base_time
    current_quarter = 1
    time_remaining = 15 * 60  # 15 minutes per quarter
    dal_score = 0
    phi_score = 0
    possession = 'DAL'
    field_position = 25
    down = 1
    distance = 10
    
    play_types = ['pass', 'run', 'punt', 'field_goal', 'touchdown', 'interception', 'sack']
    
    for i in range(100):  # 100 plays
        # Simulate play outcome
        play_type = np.random.choice(play_types, p=[0.4, 0.3, 0.1, 0.05, 0.05, 0.05, 0.05])
        
        if play_type == 'touchdown':
            yards_gained = 100 - field_position
            if possession == 'DAL':
                dal_score += 7
            else:
                phi_score += 7
            # Reset for kickoff
            possession = 'PHI' if possession == 'DAL' else 'DAL'
            field_position = 25
            down = 1
            distance = 10
        elif play_type == 'field_goal':
            yards_gained = 0
            if possession == 'DAL':
                dal_score += 3
            else:
                phi_score += 3
            possession = 'PHI' if possession == 'DAL' else 'DAL'
            field_position = 25
            down = 1
            distance = 10
        elif play_type == 'punt':
            yards_gained = 0
            possession = 'PHI' if possession == 'DAL' else 'DAL'
            field_position = np.random.randint(20, 50)
            down = 1
            distance = 10
        elif play_type == 'interception':
            yards_gained = -np.random.randint(0, 10)
            possession = 'PHI' if possession == 'DAL' else 'DAL'
            field_position = np.random.randint(20, 80)
            down = 1
            distance = 10
        elif play_type == 'sack':
            yards_gained = -np.random.randint(3, 12)
            field_position = max(1, field_position + yards_gained)
            down = min(4, down + 1)
        else:  # pass or run
            yards_gained = np.random.randint(-2, 25)
            field_position = min(100, field_position + yards_gained)
            
            if yards_gained >= distance:
                down = 1
                distance = 10
            else:
                down += 1
                distance -= yards_gained
                
            if down > 4:
                possession = 'PHI' if possession == 'DAL' else 'DAL'
                field_position = 100 - field_position
                down = 1
                distance = 10
        
        plays.append({
            'timestamp': current_time,
            'play_type': play_type,
            'possession_team': possession,
            'field_position': field_position,
            'score_home': phi_score,  # Philadelphia is home
            'score_away': dal_score,  # Dallas is away
            'quarter': current_quarter,
            'time_remaining': time_remaining,
            'down': down,
            'distance': distance,
            'yards_gained': yards_gained,
            'home_team': 'PHI',
            'away_team': 'DAL'
        })
        
        # Update game time
        current_time += timedelta(seconds=np.random.randint(30, 120))
        time_remaining -= np.random.randint(30, 120)
        
        # Handle quarter changes
        if time_remaining <= 0 and current_quarter < 4:
            current_quarter += 1
            time_remaining = 15 * 60
    
    return pd.DataFrame(plays)

def create_sample_price_data(nfl_data):
    """Create realistic sample Kalshi price data correlated with game events."""
    # Base price around 50 cents (50% probability)
    base_price = 0.50
    prices = []
    
    start_time = nfl_data['timestamp'].min()
    end_time = nfl_data['timestamp'].max() + timedelta(minutes=30)
    
    current_time = start_time
    current_price = base_price
    
    while current_time <= end_time:
        # Find nearby NFL events that might affect price
        nearby_plays = nfl_data[
            (nfl_data['timestamp'] >= current_time - timedelta(minutes=2)) &
            (nfl_data['timestamp'] <= current_time + timedelta(minutes=2))
        ]
        
        # Calculate price movement based on events
        price_change = np.random.randn() * 0.01  # Base random walk
        
        for _, play in nearby_plays.iterrows():
            # Dallas positive events increase DAL win probability
            if play['possession_team'] == 'DAL':
                if 'touchdown' in play['play_type']:
                    price_change += 0.05
                elif 'field_goal' in play['play_type']:
                    price_change += 0.02
                elif play['yards_gained'] > 15:  # Big play
                    price_change += 0.01
                elif 'sack' in play['play_type'] or 'interception' in play['play_type']:
                    price_change -= 0.03
            else:  # Philadelphia events
                if 'touchdown' in play['play_type']:
                    price_change -= 0.05
                elif 'field_goal' in play['play_type']:
                    price_change -= 0.02
                elif play['yards_gained'] > 15:  # Big play
                    price_change -= 0.01
        
        current_price = np.clip(current_price + price_change, 0.01, 0.99)
        
        # Create OHLCV data
        high_price = current_price + abs(np.random.randn() * 0.005)
        low_price = current_price - abs(np.random.randn() * 0.005)
        open_price = current_price + np.random.randn() * 0.002
        
        prices.append({
            'timestamp': current_time,
            'close_price': current_price,
            'open_price': open_price,
            'high_price': max(high_price, current_price, open_price),
            'low_price': min(low_price, current_price, open_price),
            'volume': np.random.randint(5000, 25000),
            'bid_price': current_price - 0.01,
            'ask_price': current_price + 0.01
        })
        
        current_time += timedelta(seconds=60)  # 1-minute intervals
    
    return pd.DataFrame(prices)

# Generate sample data
print("Generating sample NFL data...")
nfl_data = create_sample_nfl_data()
print(f"Created {len(nfl_data)} NFL plays")

print("Generating sample price data...")
price_data = create_sample_price_data(nfl_data)
print(f"Created {len(price_data)} price points")

# Display sample data
print("\nSample NFL Data:")
print(nfl_data.head())

print("\nSample Price Data:")
print(price_data.head())

## 2. Feature Engineering Pipeline

Now let's set up and run our comprehensive feature engineering pipeline.

In [ ]:
# Configure feature pipeline
config = FeatureConfig(
    enable_game_features=True,
    enable_technical_features=True,
    enable_momentum_features=True,
    feature_windows=[5, 10, 20],
    scaling_method='standard',
    imputation_method='mean',
    handle_outliers=True,
    enable_feature_selection=False,  # Keep all features for analysis
    min_data_points=20
)

# Initialize pipeline
pipeline = FeaturePipeline(config)

# Run feature engineering
print("Running feature engineering pipeline...")
result = pipeline.fit_transform(
    nfl_data=nfl_data,
    price_data=price_data,
    team_focus='DAL'  # Focus on Dallas Cowboys
)

print(f"\nFeature engineering completed!")
print(f"Total features extracted: {len(result.feature_names)}")
print(f"Total samples: {len(result.features_df)}")
print(f"Target columns: {result.target_columns}")
print(f"Momentum events detected: {len(result.momentum_events)}")

# Display pipeline statistics
print("\nPipeline Statistics:")
for key, value in result.pipeline_stats.items():
    if isinstance(value, dict):
        print(f"{key}:")
        for k, v in value.items():
            print(f"  {k}: {v}")
    else:
        print(f"{key}: {value}")

## 3. Feature Analysis and Validation

Let's validate our features and analyze their quality.

In [ ]:
# Initialize validator
validator = FeatureValidator()

# Run comprehensive validation
print("Running feature validation...")
validation_report = validator.validate_features(result)

print(f"\nValidation completed!")
print(f"Overall validation score: {validation_report['overall_score']:.1f}/100")

# Display key validation metrics
print("\nData Quality Metrics:")
data_quality = validation_report['data_quality']
print(f"Missing data: {data_quality['missing_data']['max_missing_percentage']:.1f}% max")
print(f"Constant features: {data_quality['constant_features']['count']}")
print(f"Duplicate features: {data_quality['duplicate_features']['count']}")

print("\nFeature Statistics:")
feature_stats = validation_report['feature_statistics']
if 'highly_skewed_features' in feature_stats:
    print(f"Highly skewed features: {len(feature_stats['highly_skewed_features'])}")

print("\nCorrelation Analysis:")
correlation = validation_report['correlation_analysis']
if 'feature_correlations' in correlation:
    high_corr = len(correlation['feature_correlations']['high_correlation_pairs'])
    print(f"High correlation pairs: {high_corr}")

print("\nMomentum Events:")
momentum_val = validation_report['momentum_validation']
if 'total_events' in momentum_val:
    print(f"Total momentum events: {momentum_val['total_events']}")
    print(f"Event types: {momentum_val.get('event_types', {})}")
    print(f"Direction distribution: {momentum_val.get('direction_distribution', {})}")

## 4. Correlation Analysis

Now let's analyze correlations between features and price movements.

In [ ]:
# Extract features and targets for correlation analysis
feature_df = result.features_df
feature_cols = [col for col in result.feature_names if col in feature_df.columns]
target_cols = [col for col in result.target_columns if col in feature_df.columns]

print(f"Analyzing correlations for {len(feature_cols)} features and {len(target_cols)} targets...")

# Calculate correlations with main target (price change)
if target_cols:
    main_target = target_cols[0]
    print(f"\nMain target: {main_target}")
    
    # Calculate correlations
    correlations = []
    for feature in feature_cols:
        if feature in feature_df.columns and main_target in feature_df.columns:
            # Remove NaN values for correlation calculation
            valid_mask = ~(feature_df[feature].isna() | feature_df[main_target].isna())
            if valid_mask.sum() > 10:  # Need at least 10 valid pairs
                corr = feature_df[feature][valid_mask].corr(feature_df[main_target][valid_mask])
                if not np.isnan(corr):
                    correlations.append({
                        'feature': feature,
                        'correlation': corr,
                        'abs_correlation': abs(corr)
                    })
    
    # Sort by absolute correlation
    correlations_df = pd.DataFrame(correlations).sort_values('abs_correlation', ascending=False)
    
    print(f"\nTop 20 Features by Correlation with {main_target}:")
    print(correlations_df.head(20).to_string(index=False))
    
    # Plot top correlations
    plt.figure(figsize=(12, 8))
    top_20 = correlations_df.head(20)
    
    colors = ['red' if x < 0 else 'blue' for x in top_20['correlation']]
    bars = plt.barh(range(len(top_20)), top_20['correlation'], color=colors, alpha=0.7)
    
    plt.yticks(range(len(top_20)), top_20['feature'], fontsize=10)
    plt.xlabel('Correlation with Price Change', fontsize=12)
    plt.title(f'Top 20 Feature Correlations with {main_target}', fontsize=14, fontweight='bold')
    plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
    plt.grid(axis='x', alpha=0.3)
    
    # Add correlation values to bars
    for i, bar in enumerate(bars):
        width = bar.get_width()
        plt.text(width + 0.001 if width > 0 else width - 0.001, 
                bar.get_y() + bar.get_height()/2, 
                f'{width:.3f}', ha='left' if width > 0 else 'right', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.show()
    
    # Feature category analysis
    print("\nCorrelation Analysis by Feature Category:")
    categories = {
        'Game State': [f for f in correlations_df['feature'] if any(kw in f.lower() for kw in ['game', 'drive', 'situation', 'down', 'field'])],
        'Technical': [f for f in correlations_df['feature'] if any(kw in f.lower() for kw in ['sma', 'ema', 'rsi', 'macd', 'bb_', 'volatility'])],
        'Momentum': [f for f in correlations_df['feature'] if 'momentum' in f.lower()],
        'Volume': [f for f in correlations_df['feature'] if 'volume' in f.lower()],
        'Time Window': [f for f in correlations_df['feature'] if any(kw in f.lower() for kw in ['rolling', 'change_'])]
    }
    
    for category, features in categories.items():
        if features:
            cat_correlations = correlations_df[correlations_df['feature'].isin(features)]
            avg_abs_corr = cat_correlations['abs_correlation'].mean()
            max_abs_corr = cat_correlations['abs_correlation'].max()
            print(f"{category}: {len(features)} features, avg |corr| = {avg_abs_corr:.3f}, max |corr| = {max_abs_corr:.3f}")
else:
    print("No target columns found for correlation analysis.")

## 5. Feature Importance Analysis

Let's analyze feature importance using statistical methods.

In [ ]:
# Feature importance from validation report
if 'predictive_power' in validation_report and validation_report['predictive_power']:
    predictive_power = validation_report['predictive_power']
    
    for target_name, target_results in predictive_power.items():
        if 'error' not in target_results:
            print(f"\nFeature Importance for {target_name}:")
            
            # F-test results
            if 'f_test_results' in target_results:
                f_results = target_results['f_test_results']
                if 'top_features' in f_results:
                    print("\nTop Features by F-Test Score:")
                    f_df = pd.DataFrame(f_results['top_features'])
                    if not f_df.empty:
                        print(f_df.head(15).to_string(index=False))
                        
                        # Plot F-test scores
                        plt.figure(figsize=(12, 8))
                        top_f = f_df.head(15)
                        bars = plt.barh(range(len(top_f)), top_f['f_score'], color='skyblue', alpha=0.7)
                        plt.yticks(range(len(top_f)), top_f['feature'], fontsize=10)
                        plt.xlabel('F-Test Score', fontsize=12)
                        plt.title(f'Top 15 Features by F-Test Score ({target_name})', fontsize=14, fontweight='bold')
                        plt.grid(axis='x', alpha=0.3)
                        
                        # Add F-score values to bars
                        for i, bar in enumerate(bars):
                            width = bar.get_width()
                            plt.text(width + max(top_f['f_score']) * 0.01, 
                                    bar.get_y() + bar.get_height()/2, 
                                    f'{width:.2f}', ha='left', va='center', fontsize=9)
                        
                        plt.tight_layout()
                        plt.show()
            
            # Mutual information results
            if 'mutual_info_results' in target_results:
                mi_results = target_results['mutual_info_results']
                if 'top_features' in mi_results:
                    print("\nTop Features by Mutual Information:")
                    mi_df = pd.DataFrame(mi_results['top_features'])
                    if not mi_df.empty:
                        print(mi_df.head(15).to_string(index=False))
                        
                        # Plot mutual information scores
                        plt.figure(figsize=(12, 8))
                        top_mi = mi_df.head(15)
                        bars = plt.barh(range(len(top_mi)), top_mi['mutual_info'], color='lightcoral', alpha=0.7)
                        plt.yticks(range(len(top_mi)), top_mi['feature'], fontsize=10)
                        plt.xlabel('Mutual Information Score', fontsize=12)
                        plt.title(f'Top 15 Features by Mutual Information ({target_name})', fontsize=14, fontweight='bold')
                        plt.grid(axis='x', alpha=0.3)
                        
                        # Add MI values to bars
                        for i, bar in enumerate(bars):
                            width = bar.get_width()
                            plt.text(width + max(top_mi['mutual_info']) * 0.01, 
                                    bar.get_y() + bar.get_height()/2, 
                                    f'{width:.3f}', ha='left', va='center', fontsize=9)
                        
                        plt.tight_layout()
                        plt.show()
        break  # Only analyze first target
else:
    print("No predictive power analysis available.")

## 6. Visualization Dashboard

Let's create comprehensive visualizations of our feature engineering results.

In [ ]:
# Initialize visualizer
visualizer = FeatureVisualizer()

# Create output directory for plots
import tempfile
import os

output_dir = os.path.join(os.getcwd(), 'feature_analysis_plots')
os.makedirs(output_dir, exist_ok=True)

print(f"Creating comprehensive visualization report in {output_dir}...")

# Generate all visualizations
plot_files = visualizer.create_comprehensive_report(
    pipeline_result=result,
    validation_report=validation_report,
    output_dir=output_dir
)

print("\nGenerated visualization plots:")
for plot_name, file_path in plot_files.items():
    print(f"- {plot_name}: {file_path}")

print(f"\nAll plots saved to: {output_dir}")

## 7. Momentum Events Analysis

Let's analyze the detected momentum events and their impact on prices.

In [ ]:
if result.momentum_events:
    print(f"\nAnalyzing {len(result.momentum_events)} momentum events...")
    
    # Create momentum events DataFrame
    momentum_data = []
    for event in result.momentum_events:
        momentum_data.append({
            'timestamp': event.timestamp,
            'event_type': event.event_type.value,
            'momentum_type': event.momentum_type.value,
            'direction': event.direction.value,
            'strength': event.strength,
            'confidence': event.confidence,
            'description': event.description
        })
    
    momentum_df = pd.DataFrame(momentum_data)
    
    print("\nMomentum Events Summary:")
    print(f"Event Types: {momentum_df['event_type'].value_counts().to_dict()}")
    print(f"Momentum Types: {momentum_df['momentum_type'].value_counts().to_dict()}")
    print(f"Directions: {momentum_df['direction'].value_counts().to_dict()}")
    print(f"Average Strength: {momentum_df['strength'].mean():.3f}")
    print(f"Average Confidence: {momentum_df['confidence'].mean():.3f}")
    
    # Display strongest events
    print("\nTop 10 Strongest Momentum Events:")
    top_events = momentum_df.nlargest(10, 'strength')[['timestamp', 'event_type', 'direction', 'strength', 'description']]
    print(top_events.to_string(index=False))
    
    # Plot momentum events over time
    plt.figure(figsize=(15, 8))
    
    # Subplot 1: Price and momentum events
    plt.subplot(2, 1, 1)
    
    # Plot price if available
    if 'close_price' in feature_df.columns and 'timestamp' in feature_df.columns:
        price_data_clean = feature_df.dropna(subset=['close_price', 'timestamp'])
        plt.plot(price_data_clean['timestamp'], price_data_clean['close_price'], 
                linewidth=1, alpha=0.7, label='Close Price')
        
        # Mark momentum events on price chart
        for _, event in momentum_df.iterrows():
            color = 'green' if event['direction'] == 'bullish' else 'red'
            alpha = 0.3 + event['strength'] * 0.7
            
            # Find nearest price point
            time_diff = abs(price_data_clean['timestamp'] - event['timestamp'])
            if not time_diff.empty:
                nearest_idx = time_diff.idxmin()
                nearest_price = price_data_clean.loc[nearest_idx, 'close_price']
                
                plt.scatter(event['timestamp'], nearest_price, 
                          color=color, alpha=alpha, s=50 + event['strength'] * 100,
                          edgecolors='black', linewidth=0.5, zorder=5)
    
    plt.title('Price Movement with Momentum Events', fontsize=14, fontweight='bold')
    plt.ylabel('Price')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Subplot 2: Momentum strength timeline
    plt.subplot(2, 1, 2)
    
    # Create signed strength (positive for bullish, negative for bearish)
    signed_strength = momentum_df['strength'] * momentum_df['direction'].map({'bullish': 1, 'bearish': -1, 'neutral': 0})
    colors = ['green' if x > 0 else 'red' if x < 0 else 'gray' for x in signed_strength]
    
    plt.scatter(momentum_df['timestamp'], signed_strength, c=colors, alpha=0.7, s=80)
    plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    plt.title('Momentum Strength Timeline', fontsize=14, fontweight='bold')
    plt.ylabel('Momentum Strength (Signed)')
    plt.xlabel('Time')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
else:
    print("No momentum events detected.")

## 8. Feature Engineering Summary and Insights

Let's summarize our findings and provide actionable insights.

In [ ]:
# Generate comprehensive summary
print("=" * 80)
print("NFL TRADING FEATURE ENGINEERING SUMMARY")
print("=" * 80)

print(f"\n📊 DATA OVERVIEW:")
print(f"• NFL Plays Processed: {len(nfl_data)}")
print(f"• Price Points: {len(price_data)}")
print(f"• Aligned Records: {len(result.features_df)}")
print(f"• Features Extracted: {len(result.feature_names)}")
print(f"• Target Variables: {len(result.target_columns)}")

print(f"\n🔧 FEATURE BREAKDOWN:")
feature_breakdown = result.pipeline_stats.get('feature_breakdown', {})
for feature_type, count in feature_breakdown.items():
    print(f"• {feature_type.replace('_', ' ').title()}: {count}")

print(f"\n⚡ MOMENTUM ANALYSIS:")
print(f"• Total Momentum Events: {len(result.momentum_events)}")
if result.momentum_events:
    momentum_summary = MomentumDetector().get_momentum_summary(result.momentum_events)
    print(f"• Average Event Strength: {momentum_summary['avg_strength']:.3f}")
    print(f"• Average Confidence: {momentum_summary['avg_confidence']:.3f}")
    print(f"• Event Type Distribution: {momentum_summary['momentum_types']}")
    print(f"• Direction Distribution: {momentum_summary['directions']}")

print(f"\n✅ VALIDATION RESULTS:")
print(f"• Overall Validation Score: {validation_report['overall_score']:.1f}/100")
print(f"• Data Quality: {100 - validation_report['data_quality']['missing_data']['max_missing_percentage']:.1f}% complete")
if 'stability_tests' in validation_report:
    stability_pct = validation_report['stability_tests'].get('stability_percentage', 0)
    print(f"• Feature Stability: {stability_pct:.1f}% stable")

print(f"\n🎯 TOP PREDICTIVE FEATURES:")
if 'correlations_df' in locals() and not correlations_df.empty:
    top_5_features = correlations_df.head(5)
    for i, row in top_5_features.iterrows():
        print(f"• {row['feature']}: {row['correlation']:+.3f} correlation")

print(f"\n📈 KEY INSIGHTS:")
insights = []

# Analyze feature types effectiveness
if 'correlations_df' in locals() and not correlations_df.empty:
    # Find best performing category
    game_features = correlations_df[correlations_df['feature'].str.contains('game|drive|situation', case=False, na=False)]
    tech_features = correlations_df[correlations_df['feature'].str.contains('sma|ema|rsi|macd', case=False, na=False)]
    momentum_features = correlations_df[correlations_df['feature'].str.contains('momentum', case=False, na=False)]
    
    if not game_features.empty:
        avg_game_corr = game_features['abs_correlation'].mean()
        insights.append(f"Game state features show average |correlation| of {avg_game_corr:.3f}")
    
    if not tech_features.empty:
        avg_tech_corr = tech_features['abs_correlation'].mean()
        insights.append(f"Technical indicators show average |correlation| of {avg_tech_corr:.3f}")
    
    if not momentum_features.empty:
        avg_momentum_corr = momentum_features['abs_correlation'].mean()
        insights.append(f"Momentum features show average |correlation| of {avg_momentum_corr:.3f}")

# Momentum insights
if result.momentum_events:
    game_momentum_events = [e for e in result.momentum_events if e.momentum_type.value == 'game_momentum']
    price_momentum_events = [e for e in result.momentum_events if e.momentum_type.value == 'price_momentum']
    
    if game_momentum_events:
        insights.append(f"Detected {len(game_momentum_events)} game momentum events")
    if price_momentum_events:
        insights.append(f"Detected {len(price_momentum_events)} price momentum events")

# Data quality insights
missing_pct = validation_report['data_quality']['missing_data']['max_missing_percentage']
if missing_pct < 5:
    insights.append("Excellent data quality with minimal missing values")
elif missing_pct < 15:
    insights.append("Good data quality with manageable missing values")
else:
    insights.append("Data quality concerns - consider additional preprocessing")

for i, insight in enumerate(insights, 1):
    print(f"• {insight}")

print(f"\n🚀 RECOMMENDATIONS:")
recommendations = [
    "Use top correlated features for initial model training",
    "Monitor momentum events for real-time trading signals",
    "Consider ensemble methods combining different feature types",
    "Implement feature selection based on statistical significance",
    "Validate model performance with out-of-sample data"
]

for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")

print(f"\n" + "=" * 80)
print(f"Feature engineering pipeline completed successfully!")
print(f"Generated visualizations saved to: {output_dir}")
print("=" * 80)

## 9. Next Steps

Based on this analysis, here are the recommended next steps:

1. **Model Development**: Use the top-correlated features to build predictive models
2. **Real-time Implementation**: Deploy momentum detection for live trading signals
3. **Feature Selection**: Implement automated feature selection based on performance metrics
4. **Validation**: Test the pipeline on additional games and market conditions
5. **Optimization**: Fine-tune feature engineering parameters based on results

The feature engineering pipeline provides a solid foundation for NFL trading strategies, combining game state analysis, technical indicators, and momentum detection to create a comprehensive set of predictive features.